# 06 · Mejorar el modelo con fotos de celular

## Por qué este cuaderno
El análisis mostró que el modelo **acierta el 100 % de las fotos propias con las que entrenó y detecta 2 de 13 de las nuevas**:
memoriza esas paredes. Lo que le falta no es más épocas ni otra arquitectura, son **muchas paredes distintas** en el formato en
que se va a usar (fotos de celular). Este cuaderno retoma tu modelo y lo ajusta con eso.

## Qué tenés que haber preparado
```
data/campo/con/<pared_1>/ foto.jpg ...    ← paredes CON grieta, una subcarpeta por pared
data/campo/sin/<pared_1>/ foto.jpg ...    ← paredes SIN grieta, una subcarpeta por pared
```
* **Mínimo 4 paredes por clase; lo recomendable, 10 o más**, con 5 a 8 fotos de cada una (distinto ángulo y distancia).
* Paredes **distintas** a las de `fotos_propias` y `problemas`. Esas dos carpetas son la PRUEBA y no se tocan.

## Cómo se evita engañarse
Las paredes nuevas se reparten **por pared entera**: algunas quedan para validar mientras entrena. Así hay una señal honesta de si
generaliza a paredes que no vio, y el entrenamiento se detiene cuando esa señal deja de mejorar.

In [ ]:
# ===================== CELDA DE ARRANQUE: no hay que editar nada =====================
# Monta tu Drive, encuentra sola la carpeta del proyecto y Surface Crack, y deja todo listo.
from google.colab import drive
drive.mount("/content/drive")

import glob, os, sys
_raiz = "/content/drive/MyDrive"
_cands = sorted(glob.glob(f"{_raiz}/*/src/datos.py") + glob.glob(f"{_raiz}/*/*/src/datos.py"),
                key=os.path.getmtime, reverse=True)
if not _cands:
    raise FileNotFoundError("No encontré la carpeta del proyecto NUEVO en tu Drive (busco una carpeta que tenga src/datos.py). "
                            "Subí la carpeta del zip nuevo a tu Drive y volvé a ejecutar esta celda.")
PROYECTO = os.path.dirname(os.path.dirname(_cands[0]))
os.chdir(PROYECTO)
sys.path.insert(0, PROYECTO)
print("Proyecto:", PROYECTO)
!pip -q install scikit-image scikit-learn pandas matplotlib

from src import colab
RUTA_DATOS = colab.preparar(PROYECTO, _raiz)       # Surface Crack (zip -> disco local, o carpeta en Drive)

import json, shutil, zipfile
from pathlib import Path
import numpy as np, pandas as pd
from src import config, datos, evaluar, entrenar, modelo, mejora, mosaico

## Paso 1 · Armar los datos
Imprime cuántas fotos y paredes hay en cada parte. Si faltan paredes, el mensaje dice cuántas hacen falta.

In [ ]:
tr, va, resumen = mejora.preparar_mejora(PROYECTO, modelo_base="v2_mobilenet")
display(resumen)
print("Imágenes por época:", len(tr), "| validación (paredes nuevas):", len(va))

## Paso 2 · Ajustar el modelo
Retoma `v2_mobilenet`, libera las últimas 60 capas con tasa baja (1e-4) y entrena hasta 10 épocas, deteniéndose cuando la
validación en paredes nuevas deja de mejorar. **Necesita GPU** (*Entorno de ejecución → Cambiar tipo de entorno → T4 GPU*).
Si no hay GPU disponible, tarda mucho: no lo corras en CPU.

In [ ]:
import tensorflow as tf
print("GPU:", tf.config.list_physical_devices("GPU"))
meta = entrenar.ajustar_dominio("v2_mobilenet", "v3_celular", tr, va, epocas=10)
print(meta)

## Paso 3 · Medirlo sobre tus fotos de prueba (que el modelo NUNCA vio)
Compara el modelo anterior (v2) con el ajustado (v3) sobre tus 29 fotos, con las mismas cuatro formas de preguntar. Lo que
importa: **¿cuántas de las 13 grietas detecta, con cuántas falsas alarmas?**

In [ ]:
from tensorflow import keras
from PIL import Image
fp = pd.read_csv("results/fotos_propias_analisis.csv").dropna(subset=["etiqueta"])
filas_resumen = []
for nombre in ("v2_mobilenet", "v3_celular"):
    inf = modelo.modelo_inferencia(keras.models.load_model(f"models/{nombre}/modelo.keras"), "mobilenetv2")
    pred = entrenar.predictor(inf)
    filas = []
    for _, f in fp.iterrows():
        foto = np.asarray(Image.open(f"data/fotos_propias/{f.archivo}").convert("RGB"))
        entera = np.asarray(Image.fromarray(foto).resize((config.IMG, config.IMG)), dtype=np.float32)[None]
        fila = {"archivo": f.archivo, "etiqueta": int(f.etiqueta), "p_entera": float(pred(entera)[0])}
        for esc in (0.5, 0.25):
            fila[f"p_esc_{esc}"] = float(mosaico.clasificar_mosaico(foto, pred, escalas=(esc,))["prob_max"])
        filas.append(fila)
    diag = pd.DataFrame(filas)
    diag.to_csv(f"results/diagnostico_{nombre}.csv", index=False)
    r = evaluar.resumen_diagnostico(diag)
    r.insert(0, "modelo", nombre)
    filas_resumen.append(r)
comparacion = pd.concat(filas_resumen)
display(comparacion)
comparacion.to_csv("results/comparacion_v2_v3_fotos_propias.csv", index=False)

## Paso 4 · ¿Olvidó lo que sabía? (Surface Crack)
Si el ajuste destruyó lo aprendido, la exactitud en Surface Crack cae. Se mide con la validación **original** del modelo.
Una caída pequeña es aceptable si en las fotos de celular se detectan muchas más grietas.

In [ ]:
va_sc = pd.read_csv("models/v2_mobilenet/particion_validacion.csv")
va_sc = pd.concat([va_sc[va_sc.etiqueta == e].sample(500, random_state=1) for e in (0, 1)])
for nombre in ("v2_mobilenet", "v3_celular"):
    inf = modelo.modelo_inferencia(keras.models.load_model(f"models/{nombre}/modelo.keras"), "mobilenetv2")
    p = entrenar.predecir_rutas(entrenar.predictor(inf), va_sc["ruta"].tolist())
    r = evaluar.metricas(va_sc["etiqueta"].to_numpy(), p)
    print(f"{nombre}: exactitud {r['exactitud']:.4f} · recall {r['recall']:.4f} · falsos neg. {r['fn']} · falsos pos. {r['fp']}")

## Paso 5 · Exportar para el celular

In [ ]:
tam = modelo.exportar_tflite(keras.models.load_model("models/v3_celular/modelo.keras"), "models/v3_celular/modelo.tflite", "mobilenetv2")
print(f"modelo.tflite: {tam/1e6:.2f} MB")